In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_regression, f_regression, VarianceThreshold

from xgboost import XGBRegressor

from pathlib import Path
from google.colab import drive

CV_FOLDS = 5
TREE_SEARCH_ITER = 25
RANDOM_STATE = 142

@dataclass(frozen=True)
class RegressionTask:
    name: str
    target_column: str

@dataclass(frozen=True)
class SearchConfig:
    model_name: str
    pipeline: Pipeline
    params: dict[str, list]
    search_kind: str
    feature_selection_method: str = 'mi'


def _load_dataset():
  drive.mount('/content/drive')
  df = pd.read_excel('/content/drive/MyDrive/МИФИ Машинное обучение/Классическое машинное обучение/Данные_для_курсовой_Классическое_МО.xlsx')
  df = df.drop(columns=['Unnamed: 0'])
  return df

def _split_data(X, y, stratify=None, test_size=0.2):
    return train_test_split(X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=stratify)

def _make_results_subdir(subdir_name):
  base_dir = Path("results")
  base_dir.mkdir(exist_ok=True)
  exp_dir = base_dir / subdir_name
  exp_dir.mkdir(exist_ok=True)
  return exp_dir

def _build_search_configs() -> list['SearchConfig']:
    K_BEST = [15, 25, 50]
    ridge_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_regression)),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ])

    lasso_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_regression)),
        ("model", Lasso(random_state=RANDOM_STATE, max_iter=5000)),
    ])

    elastic_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_regression)),
        ("model", ElasticNet(random_state=RANDOM_STATE, max_iter=5000)),
    ])

    rf_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_regression)),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ])

    gbr_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_regression)),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ])

    xgb_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0)),
        ("selector", SelectKBest(score_func=mutual_info_regression)),
        ("model", XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)),
    ])

    return [
        SearchConfig(
            model_name="Ridge",
            pipeline=ridge_pipeline,
            params={
                "selector__k": K_BEST,
                "model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="Lasso",
            pipeline=lasso_pipeline,
            params={
                "selector__k": K_BEST,
                "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="ElasticNet",
            pipeline=elastic_pipeline,
            params={
                "selector__k": K_BEST,
                "model__alpha": [0.001, 0.01, 0.1, 1.0],
                "model__l1_ratio": [0.2, 0.5, 0.8, 1.0],
            },
            search_kind="grid"
        ),
        SearchConfig(
            model_name="RandomForest",
            pipeline=rf_pipeline,
            params={
                "selector__k": K_BEST,
                "model__n_estimators": [200, 400, 600],
                "model__max_depth": [None, 10, 20, 30],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", 0.5, 0.8]
            },
            search_kind="random"
        ),
        SearchConfig(
            model_name="GradientBoosting",
            pipeline=gbr_pipeline,
            params={
                "selector__k": K_BEST,
                "model__n_estimators": [100, 200, 300, 500],
                "model__learning_rate": [0.02, 0.05, 0.1, 0.2],
                "model__max_depth": [2, 3, 4, 5],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__subsample": [0.7, 0.85, 1.0]
            },
            search_kind="random"
        ),
        SearchConfig(
          model_name="XGBoost",
          pipeline=xgb_pipeline,
          params={
              "selector__k": K_BEST,
              "model__n_estimators": [100, 300, 500],
              "model__learning_rate": [0.01, 0.05, 0.1],
              "model__max_depth": [3, 5, 7],
              "model__subsample": [0.7, 0.85, 1.0]
          },
          search_kind="random")
    ]

def _prepare_task_data(task: RegressionTask) -> tuple[pd.DataFrame, pd.Series]:
    df = _load_dataset().copy()

    targets = ['IC50, mM', 'CC50, mM', 'SI']
    for i, col in enumerate(targets):
      log_col = f'log_{col}'
      df[log_col] = np.log10(df[col])

    exclude = ['IC50, mM', 'CC50, mM', 'SI', 'log_IC50, mM', 'log_CC50, mM', 'log_SI']
    feature_cols = [c for c in df.columns if c not in exclude]

    X = df[feature_cols].copy()
    y = df[task.target_column].copy()
    X = X.loc[:, X.std() > 0]
    return X, y


def _save_residuals_plot(y_true: np.ndarray, y_pred: np.ndarray, path: str) -> None:
    residuals = y_true - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].scatter(y_pred, residuals, alpha=0.5)
    axes[0].axhline(y=0, color='r', linestyle='--')
    axes[0].set_xlabel("Predicted values")
    axes[0].set_ylabel("Residuals")
    axes[0].set_title("Residuals vs Predicted")

    sns.histplot(residuals, kde=True, ax=axes[1])
    axes[1].set_title("Distribution of Residuals")

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def _save_pred_vs_true(y_true: np.ndarray, y_pred: np.ndarray, path: str) -> None:
    plt.figure(figsize=(5,5))
    plt.scatter(y_true, y_pred, alpha=0.5)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, 'r--')
    plt.xlabel("True values")
    plt.ylabel("Predicted values")
    plt.title("Predicted vs True (log scale)")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()


def run_regression_task(task: RegressionTask, verbose=False) -> pd.DataFrame:
    if (verbose):
      display(f"start task {task.name}")

    X, y = _prepare_task_data(task)
    X_train, X_test, y_train, y_test = _split_data(X, y)

    cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    exp_dir = _make_results_subdir(f"regression_{task.name}")

    records: list[dict[str, float | str]] = []
    best_model_name = ""
    best_r2 = -np.inf
    best_predictions: np.ndarray | None = None

    for config in _build_search_configs():
        if config.search_kind == "grid":
            search = GridSearchCV(
                estimator=config.pipeline,
                param_grid=config.params,
                scoring="r2",
                cv=cv,
                n_jobs=-1,
                refit=True,
            )
        else:
            search = RandomizedSearchCV(
                estimator=config.pipeline,
                param_distributions=config.params,
                n_iter=TREE_SEARCH_ITER,
                scoring="r2",
                cv=cv,
                n_jobs=-1,
                random_state=RANDOM_STATE,
                refit=True,
            )

        if (verbose):
          display(f'fit model :{config.model_name}')

        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        y_pred = best_model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        record = {
            "task": task.name,
            "model": config.model_name,
            "cv_r2_mean": float(search.best_score_),
            "test_mae": mae,
            "test_rmse": rmse,
            "test_r2": r2,
            "best_params": json.dumps(search.best_params_, ensure_ascii=False)
        }

        records.append(record)

        if r2 > best_r2:
            best_r2 = r2
            best_model_name = config.model_name
            best_predictions = y_pred

        if (verbose):
          display(record)

    result_df = pd.DataFrame(records).sort_values(by="test_r2", ascending=False)
    result_df.to_csv(exp_dir/f"{task.name} model_comparison.csv", index=False)

    if best_predictions is None:
        raise RuntimeError("Не удалось получить предсказания лучшей модели.")

    _save_pred_vs_true(y_test, best_predictions, str(exp_dir/f"{task.name} {best_model_name} pred_vs_true.png"))
    _save_residuals_plot(y_test, best_predictions, str(exp_dir/f"{task.name} {best_model_name} residuals.png"))

    pred_table = pd.DataFrame({
        "y_true": y_test.to_numpy(),
        "y_pred": best_predictions,
    })

    return result_df